# Tree-CoT-RT for ASQP ? Kaggle Offline

Attach the offline bundle dataset and the Qwen2.5-7B-Instruct Kaggle Model, enable GPU, and run all cells. Internet is not used. Results are saved under `/kaggle/working/tree_cot_rt_outputs/`.


In [ ]:
# Kaggle Input paths; source/data are discovered recursively under /kaggle/input.
PROJECT_DIR = "/kaggle/input/datasets/tuyennguyen21pt/kaggle-offline-tree/project"
DATA_DIR = None
OUTPUT_DIR = "/kaggle/working/tree_cot_rt_outputs"
MODEL_DIR = "/kaggle/input/models/dipamc77/qwen2.5-7b-instruct/transformers/default/1/Qwen2.5-7B-Instruct"

# Paper defaults. Reduce these and set MAX_* for a smoke test only.
MODEL_NAME = MODEL_DIR
FINAL_MODEL_NAME = MODEL_DIR
CHAINS_PER_TYPE = 10
GRPO_STEPS = 10
SELECTED_CHAINS_PER_TYPE = 2
FINAL_STEPS = 10
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 5e-5
MAX_LENGTH = 1024
MAX_NEW_TOKENS = 1024
INFERENCE_SAMPLES = 5
MAX_TRAIN_RECORDS = None
MAX_TEST_RECORDS = None
SEED = 444


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Prevent Hugging Face code from attempting any network access.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["WANDB_DISABLED"] = "true"

input_root = Path("/kaggle/input")
if not input_root.is_dir():
    raise FileNotFoundError("/kaggle/input was not found; run this notebook on Kaggle.")


def find_project(root):
    if PROJECT_DIR:
        candidate = Path(PROJECT_DIR)
        if (candidate / "src" / "tree_cot_rt" / "training.py").is_file():
            return candidate
        raise FileNotFoundError(f"PROJECT_DIR does not contain src/tree_cot_rt/training.py: {candidate}")
    # Kaggle datasets may omit pyproject.toml or expose only the uploaded source tree.
    # The package directory is enough because the notebook imports from project/src.
    matches = [p.parent.parent for p in root.rglob("tree_cot_rt") if p.is_dir() and (p / "training.py").is_file()]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one Tree-CoT-RT source package under /kaggle/input, found {matches}. Set PROJECT_DIR to its project directory (the parent of src/).")
    return matches[0]


def find_data(root):
    if DATA_DIR:
        candidate = Path(DATA_DIR)
        if all((candidate / f"{split}.jsonl").is_file() for split in ("train", "dev", "test")):
            return candidate
        raise FileNotFoundError(f"DATA_DIR must contain train.jsonl, dev.jsonl, and test.jsonl: {candidate}")
    matches = [p.parent for p in root.rglob("train.jsonl") if all((p.parent / f"{s}.jsonl").is_file() for s in ("dev", "test"))]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one train/dev/test dataset under /kaggle/input, found {matches}. Set DATA_DIR.")
    return matches[0]


project_dir = find_project(input_root)
data_dir = find_data(input_root)
model_dir = Path(MODEL_DIR)
if not model_dir.is_dir():
    raise FileNotFoundError(f"Attached Kaggle model directory not found: {model_dir}. Attach Qwen2.5-7B-Instruct or set MODEL_DIR.")
wheelhouse = next((p / "wheelhouse" for p in input_root.rglob("offline_bundle") if (p / "wheelhouse").is_dir()), None)
if wheelhouse is None:
    raise FileNotFoundError("Could not locate offline_bundle/wheelhouse under /kaggle/input.")
sys.path.insert(0, str(project_dir / "src"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Project:", project_dir)
print("Data:", data_dir)
print("Wheelhouse:", wheelhouse)
print("Model:", model_dir)
print("Output:", OUTPUT_DIR)

# Install only pinned local wheels. --no-deps avoids pip resolver conflicts; Kaggle supplies torch and common runtime dependencies.
wheel_names = ("huggingface_hub==0.28.1", "tokenizers==0.21.0", "transformers==4.48.3", "accelerate==1.15.0", "peft==0.14.0", "safetensors==0.6.2")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-index", "--no-deps", "--find-links", str(wheelhouse), *wheel_names])
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before running the full training pipeline.")


In [ ]:
from tree_cot_rt.data import read_jsonl
from tree_cot_rt.training import train

train_records = read_jsonl(data_dir / "train.jsonl")
dev_records = read_jsonl(data_dir / "dev.jsonl")
test_records = read_jsonl(data_dir / "test.jsonl")
if MAX_TRAIN_RECORDS is not None:
    train_records = train_records[:MAX_TRAIN_RECORDS]
if MAX_TEST_RECORDS is not None:
    test_records = test_records[:MAX_TEST_RECORDS]
print(f"Records: train={len(train_records)}, dev={len(dev_records)}, test={len(test_records)}")

training_metrics = train(
    train_records,
    dev_records,
    OUTPUT_DIR,
    model_name=MODEL_NAME,
    final_model_name=FINAL_MODEL_NAME,
    chains_per_type=CHAINS_PER_TYPE,
    grpo_steps=GRPO_STEPS,
    selected_chains_per_type=SELECTED_CHAINS_PER_TYPE,
    final_steps=FINAL_STEPS,
    batch_size=BATCH_SIZE,
    gradient_accumulation=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    max_length=MAX_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SEED,
)
training_metrics

In [ ]:
from tree_cot_rt.cli import exact_f1
from tree_cot_rt.data import write_jsonl
from tree_cot_rt.training import predict

predictions = predict(
    test_records,
    Path(OUTPUT_DIR) / "final",
    base_model=FINAL_MODEL_NAME,
    samples=INFERENCE_SAMPLES,
    max_new_tokens=MAX_NEW_TOKENS,
)
prediction_path = Path(OUTPUT_DIR) / "test_predictions.jsonl"
write_jsonl(prediction_path, predictions)
gold = {row.instance_id: [quad.as_dict() for quad in row.quads] for row in test_records}
predicted = {row["instance_id"]: row["quadruples"] for row in predictions}
evaluation = exact_f1(gold, predicted)
evaluation_path = Path(OUTPUT_DIR) / "evaluation_metrics.json"
evaluation_path.write_text(__import__("json").dumps(evaluation, indent=2) + "\n", encoding="utf-8")
print("Exact quadruple micro-F1:", evaluation)
print("Predictions:", prediction_path)
print("Training log:", Path(OUTPUT_DIR) / "training_metrics.json")